# Healthcare FHIR Integration : HL7 v2 vers FHIR

**Entrée** : `hl7/sample_message.hl7`, message d'admission ADT^A01 **fictif**  
**Sortie** : un dictionnaire FHIR `Patient` en mémoire (aucun fichier patient n'est écrit)  
**Stack** : Python (bibliothèque standard), module `hl7/hl7_to_fhir.py`

---

### Objectif

Beaucoup d'hôpitaux émettent encore du HL7 v2 (segments séparés par `|`), alors que les systèmes récents utilisent FHIR. Ce notebook montre la traduction du segment `PID` (identifiant, nom, date de naissance, genre) vers un `Patient` FHIR.

### Fonctionnement

1. **Lecture** du message et isolement du segment `PID`.
2. **Mapping déterministe**, sans LLM : une date ou un genre mal converti dans un dossier patient n'est pas acceptable. `PID-3` devient `Patient.identifier` (et non `Patient.id`), `PID-5` le nom, `PID-7` la date de naissance, `PID-8` le genre.
3. **Cas limites explicites** : dates partielles (`1992`, `199204`) conservées, date impossible (`20260231`) rejetée, code de genre inattendu converti en `unknown` avec un avertissement.

### Limite assumée

Seul le segment `PID` d'un ADT simulé est traité. Ce n'est pas un moteur d'intégration HL7 complet.

## 1. Chargement du message

On lit le message ADT fictif `hl7/sample_message.hl7`.


In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hl7.hl7_to_fhir import (
    convert_birth_date,
    convert_gender,
    find_pid_segment,
    parse_pid,
    read_hl7_message,
)

message_path = PROJECT_ROOT / "hl7" / "sample_message.hl7"
message = read_hl7_message(message_path)
print(f"Message chargé : {len(message.splitlines())} segments")
print(message)

Message chargé : 2 segments
MSH|^~\&|HOSPITAL_A|PARIS|CONNECTOR|APP|202609182200||ADT^A01|MSG00001|P|2.5
PID|1||PAT12345^^^HOSPITAL_A^MR||MARTIN^Julie||19920403|F


> **Résultat.** Le message est lu correctement : il contient 2 segments, séparés par un retour à la ligne, dont les champs sont séparés par `|`. Il est fictif, on peut donc l'afficher ci-dessus.

| Segment | Rôle | Contenu utile ici |
| :--- | :--- | :--- |
| `MSH` | En-tête du message | Émetteur `HOSPITAL_A` (Paris), destinataire `CONNECTOR`, date d'envoi `202609182200`, type `ADT^A01` (admission), version 2.5 |
| `PID` | Identité administrative du patient | Identifiant `PAT12345` (numéro de dossier, autorité `HOSPITAL_A`), `MARTIN^Julie`, né le `19920403`, genre `F` |

Les valeurs sont affichées telles qu'elles arrivent, sans conversion. Seul le `PID` est converti dans la suite (date de naissance, genre...).

## 2. Extraction du PID et conversion

Le segment PID porte l’identité administrative. Le parseur mappe `PID-3` vers `Patient.identifier`, le nom vers `Patient.name`, puis normalise la date et le genre.


In [2]:
pid_segment = find_pid_segment(message)
fhir_patient = parse_pid(pid_segment)

print(json.dumps(fhir_patient, indent=2, ensure_ascii=False))


{
  "resourceType": "Patient",
  "identifier": [
    {
      "value": "PAT12345"
    }
  ],
  "name": [
    {
      "family": "MARTIN",
      "given": [
        "Julie"
      ]
    }
  ],
  "gender": "female",
  "birthDate": "1992-04-03"
}


> **Résultat.** Une ressource `Patient` structurée est produite. L’identifiant métier HL7 reste dans `Patient.identifier` et n’est pas confondu avec `Patient.id`, qui représente l’identifiant logique attribué par un serveur FHIR.


## 3. Cas limites : dates et genre

Dans un vrai message, la date de naissance et le genre peuvent arriver incomplets ou incorrects. On donne ici à la fonction de conversion des valeurs choisies exprès, y compris invalides, et on regarde ce qu'elle renvoie. Les deux avertissements affichés sous le code sont attendus : ils signalent les deux valeurs invalides testées.


In [3]:
date_cases = ["1992", "199204", "19920403", "20260231", ""]
gender_cases = ["F", "M", "O", "U", "X", ""]

print("Dates :", {value or "(vide)": convert_birth_date(value) for value in date_cases})
print("Genres :", {value or "(vide)": convert_gender(value) for value in gender_cases})


Date de naissance HL7 impossible : valeur ignorée


Code de sexe HL7 inattendu 'X' : normalisé en 'unknown'


Dates : {'1992': '1992', '199204': '1992-04', '19920403': '1992-04-03', '20260231': None, '(vide)': None}
Genres : {'F': 'female', 'M': 'male', 'O': 'other', 'U': 'unknown', 'X': 'unknown', '(vide)': 'unknown'}


> **Observation.** Une valeur valide est convertie, une valeur invalide est neutralisée et signalée, jamais recopiée telle quelle.

**Dates de naissance**

| Reçu | Converti | Pourquoi |
| :--- | :--- | :--- |
| `1992` | `1992` | FHIR accepte une année seule |
| `199204` | `1992-04` | année et mois |
| `19920403` | `1992-04-03` | date complète |
| `20260231` | `None` + avertissement | le 31 février n'existe pas |
| (vide) | `None` | pas d'information, pas d'avertissement |

Quand la conversion renvoie `None`, la clé `birthDate` est simplement omise de la ressource `Patient`.

**Genres**

| Reçu | Converti | Pourquoi |
| :--- | :--- | :--- |
| `F`, `M`, `O`, `U` | `female`, `male`, `other`, `unknown` | codes HL7 connus |
| `X` | `unknown` + avertissement | code inattendu : de l'information est perdue |
| (vide) | `unknown` | absence d'information, pas d'avertissement |

---

## Conclusion

Ce notebook montre la traduction du segment `PID` d'un message HL7 v2 en ressource FHIR `Patient`, sur un message d'admission fictif.

### Ce qu'on a fait

| Étape | Résultat |
| :--- | :--- |
| **Lecture** | Message de 2 segments lu : `MSH` (en-tête) et `PID` (identité du patient). |
| **Conversion** | Le `PID` devient une ressource `Patient` : identifiant `PAT12345`, nom `MARTIN`, prénom `Julie`, genre `female`, naissance `1992-04-03`. |
| **Identifiant** | L'identifiant métier HL7 va dans `Patient.identifier`, pas dans `Patient.id`, qui est réservé à l'identifiant attribué par un serveur FHIR. |
| **Cas limites** | Dates partielles conservées, date impossible neutralisée, code de genre inconnu remplacé par `unknown` avec avertissement. |

### Limites

- Seul le segment `PID` d'un message ADT simulé est traité.
- Les séparateurs personnalisés, les champs répétés (`~`) et les caractères d'échappement HL7 ne sont pas gérés.
- L'autorité d'attribution de l'identifiant (`HOSPITAL_A`) est ignorée pour l'instant.